# 정해진 제안을 검사해 봐요

실제 AI 대신 문장 다섯 개에 대한 답을 미리 적어 둔 표를 써요. API 키나 유료 계정은 필요하지 않아요.

실행하면 통과 3건과 거부 2건이 나와요. 표에 없는 문장은 처리하지 못해요.

## 시작하기

1. 위 메뉴에서 **파일 → Drive에 사본 저장**을 눌러요. 내 사본에 답을 남길 수 있어요.
2. 첫 코드 칸 왼쪽의 **▶**를 눌러요. `준비 완료`가 나오면 다음 칸으로 가요.
3. 수업 중에는 안내한 칸만 실행해요. **Shift+Enter**도 같은 칸을 실행하는 방법이에요.

`Cell`은 코드나 설명이 들어 있는 칸이에요. 런타임이 초기화되어 파일이나 변수가 사라졌다면 첫 칸부터 다시 실행해요. 단순히 브라우저를 다시 여는 것과는 달라요.

코드 앞의 `#`는 설명이에요. 실행되지 않아요. `import`는 다른 파일의 이름을 가져오고, 줄 앞의 `!`는 Python 대신 터미널 명령을 실행해요.


In [ ]:
# ── 첫 Cell · 준비 ──────────────────────────────────────────
# Colab은 구글이 빌려주는 컴퓨터입니다. 새로 켤 때마다 빈 컴퓨터이므로
# 필요한 파일을 GitHub에서 받아 와야 합니다. 이 Cell이 그 일을 합니다.
#
# 실행하는 법: 이 Cell을 클릭한 뒤 왼쪽 ▶ 버튼을 누르거나 Shift+Enter.
# 위 메뉴 [런타임 → 모두 실행]을 누르면 위에서 아래로 전부 실행됩니다.

import os, sys          # import = 파이썬에 이미 들어 있는 도구 상자를 꺼내는 일

# 줄 앞의 ! 는 "파이썬이 아니라 터미널 명령"이라는 표시입니다.
# git clone = GitHub에 있는 폴더를 통째로 이 컴퓨터로 복사하는 명령.
if not os.path.isdir("jnu-llmops-precourse-day2"):      # 이미 받았으면 건너뜁니다
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day2.git
if not os.path.isdir("jnu-llmops-precourse-day3"):
    !git clone -q https://github.com/GoBeromsu/jnu-llmops-precourse-day3.git

# 파이썬이 import 할 파일을 찾는 폴더 목록에 어제의 solution 폴더를 넣습니다.
# 이 줄이 없으면 아래 Cell의 from order import Order 가 파일을 못 찾습니다.
sys.path.insert(0, "jnu-llmops-precourse-day2/solution")
print("준비 완료 · 왼쪽 파일 탭에 두 폴더가 생겼는지 확인하세요")

## 1. 주문 검사 함수를 준비해요

어제 함수의 완료본이에요. 내 노트북을 다 끝내지 못했어도 여기서 시작할 수 있어요. 다음 칸을 실행하면 함수가 준비돼요.

In [ ]:
import json
from catalog import MENU
from order import Order
from pricing import calculate_bill

REQUIRED_KEYS = ("order_id", "items", "is_student")

def check_order(record):
    for key in REQUIRED_KEYS:                       # 1. 필수 Key
        if key not in record:
            return False, f"필수 Key 없음: {key}"
    for item in record["items"]:
        if item["menu_name"] not in MENU:           # 2. 허용 메뉴
            return False, f"없는 메뉴: {item['menu_name']}"
        quantity = item["quantity"]
        if type(quantity) is not int or not 1 <= quantity <= 10:   # 3. 수량 범위
            return False, f"수량 범위 밖: {quantity!r}"
    return True, ""


## 2. 다섯 문장의 제안을 확인해요

`suggest_order`는 모델이 아니라 답이 정해진 표예요. 다음 칸을 실행하면 문장마다 통과 여부와 거부 이유가 나와요.

거부된 두 문장을 찾아보세요. 메뉴 이름과 수량 중 무엇이 맞지 않나요?

In [ ]:
SUGGESTIONS = {
    "라떼 두 잔, 학생이에요": {"order_id": "S01", "items": [{"menu_name": "카페라떼", "quantity": 2}], "is_student": True},
    "아메리카노 하나요":       {"order_id": "S02", "items": [{"menu_name": "아메리카노", "quantity": 1}], "is_student": False},
    "초코라떼 셋, 학생 할인":  {"order_id": "S03", "items": [{"menu_name": "초코라떼", "quantity": 3}], "is_student": True},
    "녹차라떼 한 잔":          {"order_id": "S04", "items": [{"menu_name": "녹차라떼", "quantity": 1}], "is_student": False},
    "라떼 두 잔":              {"order_id": "S05", "items": [{"menu_name": "카페라떼", "quantity": "두"}], "is_student": False},
}

def suggest_order(text):
    return SUGGESTIONS[text]

for text in SUGGESTIONS:
    ok, reason = check_order(suggest_order(text))
    print(f"{text!r:22} -> {ok} {reason}")

## 3. 기대한 결과와 비교해요

다음 칸의 검사가 끝나도 출력이 없으면 다섯 건이 기대와 같다는 뜻이에요.

`AssertionError`가 나오면 메시지에 있는 문장과 검사 결과를 확인하세요. 기대값을 바꾸기 전에 입력과 검사 조건을 먼저 봐요.

In [ ]:
with open("jnu-llmops-precourse-day3/data/expected_day4.json", encoding="utf-8") as f:
    expected = json.load(f)

for text, want in expected.items():
    ok, reason = check_order(suggest_order(text))
    assert ok == want, (text, ok, reason)

## 4. 거부 이유를 모아요

다음 칸을 실행하면 거부된 문장과 이유 두 건이 나와요. 같은 입력을 다시 보내기 전에 어떤 값이 잘못됐는지 확인해요.

In [ ]:
rejected = []
for text in SUGGESTIONS:
    ok, reason = check_order(suggest_order(text))
    if not ok:
        rejected.append({"text": text, "reason": reason})
print(json.dumps(rejected, ensure_ascii=False, indent=2))

## 5. 표에 없는 문장을 넣어 봐요

다음 칸은 `녹차 프라페`를 요청해요. 표에 없어서 `KeyError`가 나와요. 오류를 출력해 두었으므로 다음 설명으로 이어갈 수 있어요.

In [ ]:
try:
    suggest_order("녹차 프라페")
except KeyError as e:
    print("KeyError:", e)

## 더 해 보기 · 문장을 하나 추가해요

먼저 새 문장이 통과할지 거부될지 적어요. 그다음 `SUGGESTIONS`에 제안을, `data/expected_day4.json`에 기대값을 추가하고 검사 칸을 다시 실행해요. 두 곳 중 한 곳만 바꾸지 않았는지 확인하세요.

## 다음 사람이 시작할 수 있게 적어요

- 어느 링크를 열고 어떤 칸부터 실행하나요?
- 어떤 결과가 나오면 확인이 끝나나요?
- 이 노트북이 처리하지 못하는 것은 무엇인가요?